### Set Up Folder Structure

In [1]:
import os

# Create project folders in Colab's session storage
folders = ['/content/AES_Project/data',
           '/content/AES_Project/models',
           '/content/AES_Project/src',
           '/content/AES_Project/outputs']

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print(" Folder structure created!")

 Folder structure created!


### Install All Required Libraries
 Colab has many libraries pre-installed, so this only adds what's missing:

In [2]:
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q rouge-score bert-score
!pip install -q nltk spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 101.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### Verify GPU is Working

In [3]:
import torch
import transformers

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
print("Transformers version:", transformers.__version__)

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Transformers version: 5.3.0


You should see CUDA available: True and Tesla T4 as the GPU name.

In [4]:
import os
import pandas as pd
import gdown


os.makedirs('/content/AES_Project/data', exist_ok=True)


file_ids = {
    'training_set_rel3.csv': '1dBGoTw8F2Gcgj4_b2J94EPxzqjF16gIF', # The ASAP raw data
    'Prompt-3.csv': '1H5ZiBPfboAan_EZAlDt6mTWMMrAC7b47',
    'Prompt-4.csv': '1qvw_VgpTfROzWGa2-3CgirCyN669uuBo',
    'Prompt-5.csv': '1OS0EgyEcPcUKRthtF262l_-43b0W_ZjN',
    'Prompt-6.csv': '1Guar_mZzFtBbNEIZobQvc72rEAF0PA1L'
}

print("Downloading dataset files...")
for filename, file_id in file_ids.items():
    url = f'https://drive.google.com/uc?id={file_id}'
    output = f'/content/AES_Project/data/{filename}'
    gdown.download(url, output, quiet=False)

print("All files ready in /content/AES_Project/data/")

Downloading...
From: https://drive.google.com/uc?id=1dBGoTw8F2Gcgj4_b2J94EPxzqjF16gIF
To: /content/AES_Project/data/training_set_rel3.csv
100%|██████████| 16.4M/16.4M [00:00<00:00, 113MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1H5ZiBPfboAan_EZAlDt6mTWMMrAC7b47
To: /content/AES_Project/data/Prompt-3.csv
100%|██████████| 22.5k/22.5k [00:00<00:00, 20.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1qvw_VgpTfROzWGa2-3CgirCyN669uuBo
To: /content/AES_Project/data/Prompt-4.csv
100%|██████████| 23.7k/23.7k [00:00<00:00, 49.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1OS0EgyEcPcUKRthtF262l_-43b0W_ZjN
To: /content/AES_Project/data/Prompt-5.csv
100%|██████████| 25.3k/25.3k [00:00<00:00, 52.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Guar_mZzFtBbNEIZobQvc72rEAF0PA1L
To: /content/AES_Project/data/Prompt-6.csv
100%|██████████| 25.3k/25.3k [00:00<00:00, 32.9MB/s]

All files ready in /content/AES_Project/data/


In [5]:
asap_raw = pd.read_csv('/content/AES_Project/data/training_set_rel3.csv', encoding='latin1')

prompt3 = pd.read_csv('/content/AES_Project/data/Prompt-3.csv', encoding='latin1')

prompt4 = pd.read_csv('/content/AES_Project/data/Prompt-4.csv', encoding='latin1')

prompt5 = pd.read_csv('/content/AES_Project/data/Prompt-5.csv', encoding='latin1')

prompt6 = pd.read_csv('/content/AES_Project/data/Prompt-6.csv', encoding='latin1')


In [6]:
!pip install -U bitsandbytes accelerate transformers
#for getting. the latest available version of bitsandbytes.
#run this only if the next block of code shows importerror, typically happens when session restarts so run each time with a new runtime

In [7]:
#torch -> for building loss functions and training
#transformers -> for mistraal/gpt type models

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

#4-bit configuration for memory efficiency

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name) #turns text to numbers that the model understands
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bnb_config,
                                             device_map="auto")

#this block takes a little longer to load, maybe like ~ 6-7 minutes


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [8]:
filtered= asap_raw[asap_raw['essay_set'].isin([3,4,5,6])]
print("Total rows in prompt 3-6 is", len(filtered))
print("Columns : ", list(filtered.columns))
print("Per prompt : ", filtered['essay_set'].value_counts().sort_index().to_dict())
for i in [3,4,5,6]:
  p = pd.read_csv(f'/content/AES_Project/data/Prompt-{i}.csv')
  print(f'prompt-{i}.csv rows: {len(p)}, columns: {list(p.columns)}')

  #check for essay_id overlap in asap dataset and prompt 3-6 datasets

  overlap = filtered[filtered['essay_set'] == i]['essay_id'].isin(p['Essay ID']).sum()
  print(f'Matching essay_ids: {overlap}/{len(p)}')

Total rows in prompt 3-6 is 7101
Columns :  ['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1', 'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2', 'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3', 'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1', 'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5', 'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3', 'rater3_trait4', 'rater3_trait5', 'rater3_trait6']
Per prompt :  {3: 1726, 4: 1770, 5: 1805, 6: 1800}
prompt-3.csv rows: 1726, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1726/1726
prompt-4.csv rows: 1772, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1770/1772
prompt-5.csv rows: 1805, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1805/1805
prompt-6.csv rows: 1800, columns: [

In [9]:
import pandas as pd

all_merged = []

for i in [3, 4, 5, 6]:
    #loading labels for the specific prompt
    p_labels = pd.read_csv(f'/content/AES_Project/data/Prompt-{i}.csv', encoding='latin1')

    #getting essays
    p_essays = filtered[filtered['essay_set'] == i][['essay_id', 'essay', 'domain1_score']]

    #merging on essay_id (asap uses essay_id and prompts use Essay ID)
    temp = pd.merge(p_essays, p_labels, left_on='essay_id', right_on='Essay ID')
    all_merged.append(temp)

#master dataframe
df_master = pd.concat(all_merged, ignore_index=True)

#standardizing column names
df_master = df_master.rename(columns={
    'domain1_score': 'holistic',
    'Content': 'content',
    'Prompt Adherence': 'adherence',
    'Language': 'language',
    'Narrativity': 'narrativity'
}).drop(columns=['Essay ID'])

print(f"Master Dataframe built: {len(df_master)} rows.")

Master Dataframe built: 7101 rows.


In [10]:
import torch
import torch.nn as nn
import re
from datasets import Dataset


#format essays fro model input
def format_example(row):
    text = f"""<s>[INST] Analyze the essay based on the rubric. Provide a qualitative critique first, then numerical scores.
Essay: {row['essay']} [/INST]
Reasoning: The essay demonstrates {row['content']} level depth in content. The linguistic quality is {row['language']}/10.
Critique: To improve, focus on better evidence integration and transitions.

Final Rubric Scores:
Content: {row['content']}
Language: {row['language']}
Adherence: {row['adherence']}
Narrativity: {row['narrativity']}
Holistic: {row['holistic']} </s>"""
    return {"text": text}

hf_dataset = Dataset.from_pandas(df_master)
dataset = hf_dataset.map(format_example, remove_columns = hf_dataset.column_names)



Map:   0%|          | 0/7101 [00:00<?, ? examples/s]

In [12]:
def tokenize_function(example):
  tokens = tokenizer(
      example["text"],
      truncation = True,
      padding = "max_length",
      max_length = 512
  )

  tokens["labels"] = torch.tensor(tokens["input_ids"]).clone()

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/7101 [00:00<?, ? examples/s]

In [13]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model) #preparing the quantized model for traning

#defining the lora configuration
peft_config = LoraConfig(
    r = 16, #to balance learning capacity and memory
    lora_alpha = 32, #scaling factor, its usually r * 2
    target_modules = [ #we target linear modules
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

#creating the trainable PeftModel
model = get_peft_model(model, peft_config)

#verifying if parameters are now trainable
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


In [21]:
#initializing the fine_tuning parameters

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "/content/AES_Project/outputs",
    per_device_train_batch_size = 2, #keeping memory usage low
    gradient_accumulation_steps = 4, #effectively running 8 essays at once
    learning_rate = 2e-4, #speed of leaning
    num_train_epochs = 1, #one full pass through all essays
    logging_steps = 10,
    optim = "paged_adamw_32bit", #memory efficient optimizer
    fp16 = False,
    bf16 = False,
    save_strategy = "epoch",
    report_to = "none"
)

In [22]:
!pip install -U trl

In [23]:
from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer, mlm= False)

In [24]:
#forcing the model's internal config to use float16
model.config.torch_dtype = torch.float16
model.config.use_cache = False #required for training

#every parameter is strictly float16
for name, param in model.named_parameters():
    if 'lora' in name: #only lora parameters are trainable
        param.data = param.data.to(torch.float16)

In [ ]:
#initializing the sft trainer

from trl import SFTTrainer

#connects the lora model and the formatted dataset

trainer = SFTTrainer(
    model = model,
    train_dataset = tokenized_dataset,
    data_collator = collator,
    args = training_args
)


model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

# every single part of the model to float16
model = model.to(torch.float16)

#explicitly telling the model config to forget bfloat16
model.config.torch_dtype = torch.float16

trainer.train()

Adding EOS to train dataset:   0%|          | 0/7101 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7101 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7101 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.111128
